In [1]:
import pyodbc
import psycopg2
import json
from datetime import datetime

print("=== Шаг 1: Выгрузка метаданных процедур из MSSQL ===\n")

# ============================================
# 1. Подключение к MSSQL (источник)
# ============================================
mssql_conn = pyodbc.connect(
    'DRIVER={ODBC Driver 18 for SQL Server};'
    'SERVER=db22.vra.local;'
    'DATABASE=mobile_Makeevka;'  # или другая БД, где лежат процедуры
    'UID=sa;'
    'PWD=sasa;'
    'TrustServerCertificate=yes;'  # для самоподписанного сертификата
)

print(" Подключение к MSSQL установлено")

# ============================================
# 2. Подключение к PostgreSQL (хранилище агента)
# ============================================
pg_conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="mydatabase",
    user="postgres",
    password="postgres"
)
pg_cur = pg_conn.cursor()

print(" Подключение к PostgreSQL установлено")

# ============================================
# 3. Создание таблиц в PostgreSQL (если нет)
# ============================================
pg_cur.execute("""
    CREATE TABLE IF NOT EXISTS procedures_metadata (
        id SERIAL PRIMARY KEY,
        proc_name VARCHAR(200) NOT NULL,
        proc_schema VARCHAR(100),
        param_name VARCHAR(100),
        param_type VARCHAR(50),
        param_order INT,
        is_output BOOLEAN,
        created_at TIMESTAMP DEFAULT NOW()
    )
""")

pg_cur.execute("""
    CREATE TABLE IF NOT EXISTS procedures_full_text (
        id SERIAL PRIMARY KEY,
        proc_name VARCHAR(200) UNIQUE NOT NULL,
        proc_definition TEXT,
        created_at TIMESTAMP DEFAULT NOW()
    )
""")

pg_cur.execute("""
    CREATE TABLE IF NOT EXISTS file_patterns (
        id SERIAL PRIMARY KEY,
        filename VARCHAR(200) UNIQUE NOT NULL,
        delimiter VARCHAR(5),
        column_count INT,
        column_types JSON,
        sample_values JSON,
        created_at TIMESTAMP DEFAULT NOW()
    )
""")

pg_cur.execute("""
    CREATE TABLE IF NOT EXISTS file_procedure_mapping (
        id SERIAL PRIMARY KEY,
        filename VARCHAR(200) NOT NULL,
        proc_name VARCHAR(200) NOT NULL,
        confidence FLOAT,  -- 0..1, насколько уверены
        is_verified BOOLEAN DEFAULT FALSE,
        created_by VARCHAR(50),  -- 'ai' или 'human'
        created_at TIMESTAMP DEFAULT NOW()
    )
""")

pg_conn.commit()
print(" Таблицы в PostgreSQL созданы")

# ============================================
# 4. Выгрузка метаданных процедур из MSSQL
# ============================================
print("\n--- Выгрузка параметров процедур ---")

# Очищаем старые данные
pg_cur.execute("DELETE FROM procedures_metadata")
pg_conn.commit()

# Запрос к MSSQL (через INFORMATION_SCHEMA)
cursor = mssql_conn.cursor()
cursor.execute("""
    SELECT 
        SPECIFIC_SCHEMA,
        SPECIFIC_NAME,
        PARAMETER_NAME,
        DATA_TYPE,
        ORDINAL_POSITION,
        CASE WHEN PARAMETER_MODE = 'OUT' THEN 1 ELSE 0 END as IS_OUTPUT
    FROM INFORMATION_SCHEMA.PARAMETERS
    WHERE SPECIFIC_SCHEMA = 'dbo'
    ORDER BY SPECIFIC_NAME, ORDINAL_POSITION
""")

procedures_count = 0
params_count = 0

for row in cursor.fetchall():
    pg_cur.execute("""
        INSERT INTO procedures_metadata 
        (proc_schema, proc_name, param_name, param_type, param_order, is_output)
        VALUES (%s, %s, %s, %s, %s, %s)
    """, (row[0], row[1], row[2], row[3], row[4], row[5]))
    params_count += 1
    if row[4] == 1:  # первый параметр = новая процедура
        procedures_count += 1

pg_conn.commit()
print(f" Загружено {procedures_count} процедур, {params_count} параметров")

# ============================================
# 5. Выгрузка полного текста процедур (для сложного анализа)
# ============================================
print("\n--- Выгрузка полного текста процедур ---")

# Очищаем старые данные
pg_cur.execute("DELETE FROM procedures_full_text")
pg_conn.commit()

# Получаем список уникальных процедур
pg_cur.execute("SELECT DISTINCT proc_name FROM procedures_metadata")
procs = pg_cur.fetchall()

for (proc_name,) in procs:
    try:
        # Запрос определения процедуры из MSSQL
        cursor.execute("""
            SELECT OBJECT_DEFINITION(OBJECT_ID(?))
        """, (proc_name,))
        row = cursor.fetchone()
        definition = row[0] if row[0] else ''
        
        if definition:
            pg_cur.execute("""
                INSERT INTO procedures_full_text (proc_name, proc_definition)
                VALUES (%s, %s)
                ON CONFLICT (proc_name) DO UPDATE SET proc_definition = EXCLUDED.proc_definition
            """, (proc_name, definition))
    except Exception as e:
        print(f"  ⚠ Не удалось загрузить {proc_name}: {e}")

pg_conn.commit()
print(f" Загружено полных текстов: {pg_cur.execute('SELECT COUNT(*) FROM procedures_full_text').fetchone()[0]}")

# ============================================
# 6. Проверка и статистика
# ============================================
print("\n=== СТАТИСТИКА ===")

pg_cur.execute("SELECT COUNT(*) FROM procedures_metadata")
print(f" Записей в procedures_metadata: {pg_cur.fetchone()[0]}")

pg_cur.execute("SELECT COUNT(DISTINCT proc_name) FROM procedures_metadata")
print(f" Уникальных процедур: {pg_cur.fetchone()[0]}")

pg_cur.execute("SELECT COUNT(*) FROM procedures_full_text")
print(f" Процедур с полным текстом: {pg_cur.fetchone()[0]}")

# ============================================
# 7. Пример: посмотрим первые 5 процедур
# ============================================
print("\n=== ПРИМЕР ПРОЦЕДУР (первые 5) ===")

pg_cur.execute("""
    SELECT DISTINCT proc_name 
    FROM procedures_metadata 
    ORDER BY proc_name 
    LIMIT 5
""")
for (proc_name,) in pg_cur.fetchall():
    # Получаем параметры
    pg_cur.execute("""
        SELECT param_name, param_type, param_order
        FROM procedures_metadata
        WHERE proc_name = %s
        ORDER BY param_order
    """, (proc_name,))
    params = pg_cur.fetchall()
    
    print(f"\n {proc_name}")
    for param in params:
        print(f"     - {param[0]} ({param[1]})")

# ============================================
# 8. Закрытие соединений
# ============================================
cursor.close()
mssql_conn.close()
pg_cur.close()
pg_conn.close()

print("\n Готово! Метаданные процедур сохранены в PostgreSQL")

=== Шаг 1: Выгрузка метаданных процедур из MSSQL ===



Error: ('01000', "[01000] [unixODBC][Driver Manager]Can't open lib 'ODBC Driver 18 for SQL Server' : file not found (0) (SQLDriverConnect)")